In [ ]:
# 1. 導入必要套件
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()
import numpy as np
from sklearn.datasets import fetch_lfw_people
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap

# 埋入指定資訊
print("姓名: 何柏霆, 完成日期:2026年4月8日")

# 2. 載入人臉資料集 (取用至少 60 張圖片以上的名人)
faces = fetch_lfw_people(min_faces_per_person=60)
print(f"資料集維度: {faces.data.shape}")
print(f"圖片尺寸: {faces.images[0].shape}")

# --- P.25 Eigenfaces (PCA 降維與重建) 練習 ---
print("\n執行 P.25 Eigenfaces 練習...")

# 計算 PCA (取 150 個主成分)
pca = PCA(150, svd_solver='randomized', random_state=42)
pca.fit(faces.data)

# 將資料投影到低維空間並還原
components = pca.transform(faces.data)
projected = pca.inverse_transform(components)

# 繪製結果
fig, ax = plt.subplots(2, 10, figsize=(10, 2.5),
                       subplot_kw={'xticks':[], 'yticks':[]},
                       gridspec_kw=dict(hspace=0.1, wspace=0.1))

for i in range(10):
    ax[0, i].imshow(faces.data[i].reshape(62, 47), cmap='binary_r')
    ax[1, i].imshow(projected[i].reshape(62, 47), cmap='binary_r')

ax[0, 0].set_ylabel('full-dim\ninput')
ax[1, 0].set_ylabel('150-dim\nreconstruction')
plt.show()

# --- P.45 Isomap (流形學習視覺化) 練習 ---
print("\n執行 P.45 Isomap 視覺化練習...")

# 定義繪製函式
def plot_components(data, model, images=None, ax=None, thumb_frac=0.05):
    ax = ax or plt.gca()
    proj = model.fit_transform(data)
    ax.scatter(proj[:, 0], proj[:, 1], lw=0, s=30, alpha=0.6)

    if images is not None:
        min_dist_2 = (thumb_frac * max(proj.max(0) - proj.min(0))) ** 2
        shown_images = np.array([2 * proj.max(0)])
        for i in range(data.shape[0]):
            dist = np.sum((proj[i] - shown_images) ** 2, 1)
            if np.min(dist) < min_dist_2:
                continue # 距離太近不顯示
            shown_images = np.vstack([shown_images, proj[i]])
            imagebox = plt.matplotlib.offsetbox.AnnotationBbox(
                plt.matplotlib.offsetbox.OffsetImage(images[i], cmap='binary_r'),
                proj[i])
            ax.add_artist(imagebox)

# 執行 Isomap 並繪圖
fig, ax = plt.subplots(figsize=(10, 10))
model = Isomap(n_components=2)
plot_components(faces.data, model, images=faces.images[:, ::2, ::2])
plt.show()